<a href="https://colab.research.google.com/github/patriciaruizpaz/clinical-deterioration-intelligence/blob/main/notebooks/00_setup_repo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# CELDA 1 — Configurar git y clonar el repo (limpio, sin duplicar carpetas)
# ============================================
from getpass import getpass
import os

TOKEN = getpass("Pegá tu token de GitHub: ")

!git config --global user.email "patricia.megi@gmail.com"
!git config --global user.name "patriciaruizpaz"

# Si ya existe una copia vieja del repo en este runtime, la borramos primero
if os.path.exists('/content/clinical-deterioration-intelligence'):
    !rm -rf /content/clinical-deterioration-intelligence

%cd /content
!git clone https://{TOKEN}@github.com/patriciaruizpaz/clinical-deterioration-intelligence.git
%cd clinical-deterioration-intelligence
!pwd

In [ ]:
# ============================================
# CELDA 2 — Crear la estructura de carpetas (si ya existe, no hace nada)
# ============================================
carpetas = ["data/raw", "data/processed", "notebooks", "sql", "dashboards", "docs"]

for carpeta in carpetas:
    os.makedirs(carpeta, exist_ok=True)
    open(f"{carpeta}/.gitkeep", "w").close()

print("Estructura creada ✅")

In [ ]:
# ============================================
# CELDA 3 — README actualizado (versión final, hasta Fase 8)
# ============================================
readme_contenido = """# Clinical Deterioration Intelligence

Detección temprana de deterioro clínico en las primeras 72 horas de internación.
Proyecto de portfolio — Patricia Ruiz.

## Objetivo
Identificar qué pacientes tienen mayor riesgo de deteriorar clínicamente en las
primeras 72 horas, y construir un primer modelo de alerta temprana.

## Estado
- [x] Fase 0 — Organización del repo
- [x] Fase 1 — Diseccionar el encargo
- [x] Fase 2 — Aprender el dominio
- [x] Fase 3 — Modelar el dataset (ER)
- [x] Fase 3b — Glosario de métricas de negocio
- [x] Fase 4 — Setup técnico y reproducibilidad
- [x] Fase 5 — Primer contacto con los datos
- [x] Fase 6 — Auditoría de calidad de datos
- [x] Fase 7 — Limpieza y preparación (ETL)
- [x] Fase 8 — EDA formal (univariado, bivariado, correlaciones)
- [ ] Fase 9 en adelante — pendiente

## Datos
Los archivos de datos (crudos y procesados) no se versionan en este repositorio
por su tamaño. El dataset original está disponible en:
kaggle.com/datasets/tarekmasryo/hospital-deterioration-dataset

El pipeline de limpieza (Fase 7, notebook 01_exploracion_limpieza.ipynb) documenta
paso a paso cómo se genera la versión procesada a partir de los datos crudos.

## Documentación
- `docs/contexto_dominio.md` — glosario clínico
- `docs/diccionario_datos.md` — definición de cada columna del dataset
- `docs/diagrama_er.md` — modelo entidad-relación e hipótesis de claves
- `docs/glosario_metricas.md` — fórmulas de KPIs de negocio
- `docs/auditoria_calidad.md` — hallazgos de calidad de datos (Fase 6)
- `docs/log_limpieza.md` — decisiones de limpieza aplicadas (Fase 7)

## Stack
Python (Google Colab) · SQL (BigQuery) · Power BI · GitHub
"""

with open("README.md", "w") as f:
    f.write(readme_contenido)

print("README actualizado ✅")

In [ ]:
# ============================================
# CELDA 4 — docs/contexto_dominio.md
# ============================================
contexto_dominio = '''# Contexto de dominio — Clinical Deterioration Intelligence

## Signos vitales (medidos con sensores, en tiempo real)
- Frecuencia cardíaca (heart_rate): normal 60-100 bpm. Taquicardia >100, bradicardia <60.
- Frecuencia respiratoria (respiratory_rate): normal 12-20/min. Taquipnea >24, <8 preocupante.
- SpO2 (spo2_pct): saturación de oxígeno. Normal 95-100%, preocupante <92%, grave <88%.
- Temperatura (temperature_c): normal 36-37.5°C. Alarma en ambos extremos.
- Presión arterial (systolic_bp / diastolic_bp): normal ~90-140 / 60-90.

## Variables de laboratorio (medidas con muestra de sangre, no en tiempo real)
- Lactato (lactate): marcador de falta de oxígeno en tejidos. Normal <2 mmol/L, grave >4.
- WBC (wbc_count): glóbulos blancos, respuesta inmune a infección.
- PCR / CRP (crp_level): marcador de inflamación general.
- Creatinina (creatinine): función renal. Sube con daño renal agudo.
- Score de riesgo de sepsis (sepsis_risk_score): score compuesto de vitales+labs.

## Intervenciones
- Dispositivo de oxígeno (oxygen_device): escalera de soporte, ninguno -> nasal -> mascara -> hfnc -> niv.
  El CAMBIO de escalón es tan informativo como el valor puntual de SpO2.

## Variables del paciente (una sola vez, no cambian hora a hora)
- Índice de comorbilidad (comorbidity_index): resume enfermedades crónicas de base.
- LOS (los_hours): duración total de la internación, en horas.
- Tipo de admisión (admission_type): Elective, Transfer, ED.

## Reglas de negocio implícitas
- Datos anidados por paciente: múltiples filas horarias, NO independientes entre sí.
- Autocorrelación temporal entre horas consecutivas del mismo paciente.
- Ningún paciente llega a las 72h completas (los_hours varía, promedio ~42h).

## Benchmarks de referencia
- NEWS2: revisión clínica con score >=5, urgencia con >=7.
- Tasa de deterioro sala general: ~4-10%. En este dataset: 19.4% (cohorte de alto riesgo por diseño).
'''

with open("docs/contexto_dominio.md", "w") as f:
    f.write(contexto_dominio)
print("contexto_dominio.md guardado ✅")

In [ ]:
# ============================================
# CELDA 5 — docs/diccionario_datos.md
# ============================================
diccionario_datos = '''# Diccionario de datos — Clinical Deterioration Intelligence

## patients.csv (1 fila = 1 paciente, 10.000 filas)

| Columna | Significado | Tipo / unidad |
|---|---|---|
| patient_id | Identificador único del paciente | entero |
| age | Edad del paciente | años |
| gender | Sexo del paciente | M / F |
| comorbidity_index | Índice de comorbilidad | entero, mayor = más frágil |
| admission_type | Tipo de admisión | Elective / Transfer / ED |
| baseline_risk_score | Score de riesgo basal al ingreso | decimal 0-1 |
| los_hours | Duración de la internación | horas |
| deterioration_event | Deterioro en toda la internación | 0/1 |
| deterioration_within_12h_from_admission | Deterioro en primeras 12h desde ingreso | 0/1 |
| deterioration_hour | Hora del evento de deterioro | entero, -1 = no aplica |

## vitals_timeseries.csv (1 fila = paciente + hora, PK: patient_id + hour_from_admission)

| Columna | Significado | Tipo / unidad |
|---|---|---|
| patient_id | Identificador del paciente | entero |
| hour_from_admission | Horas desde el ingreso | entero |
| heart_rate | Frecuencia cardíaca | latidos/min |
| respiratory_rate | Frecuencia respiratoria | respiraciones/min |
| spo2_pct | Saturación de oxígeno | % |
| temperature_c | Temperatura corporal | °C |
| systolic_bp | Presión arterial sistólica | mmHg |
| diastolic_bp | Presión arterial diastólica | mmHg |
| oxygen_device | Dispositivo de oxígeno | categórica |
| oxygen_flow | Flujo de oxígeno administrado | litros/min |
| mobility_score | Score de movilidad | escala numérica |
| nurse_alert | Alerta de enfermería | 0/1 |

## labs_timeseries.csv (1 fila = paciente + hora, PK: patient_id + hour_from_admission)

| Columna | Significado | Tipo / unidad |
|---|---|---|
| patient_id | Identificador del paciente | entero |
| hour_from_admission | Horas desde el ingreso | entero |
| wbc_count | Glóbulos blancos | x10³/µL |
| lactate | Lactato | mmol/L |
| creatinine | Creatinina | mg/dL |
| crp_level | Proteína C reactiva | mg/L |
| hemoglobin | Hemoglobina | g/dL |
| sepsis_risk_score | Score compuesto de riesgo de sepsis | decimal 0-1 |

## hospital_deterioration_ml_ready.csv (1 fila = observación horaria, sin patient_id)
Columnas: vitals + labs + age, gender, comorbidity_index, admission_type,
más la variable objetivo deterioration_next_12h (0/1, ventana móvil a nivel hora).

## hospital_deterioration_hourly_panel.csv (1 fila = paciente + hora)
Panel ya unido oficial: vitals + labs + TODAS las variables de patients.csv,
incluyendo deterioration_next_12h Y deterioration_within_12h_from_admission juntas.
'''

with open("docs/diccionario_datos.md", "w") as f:
    f.write(diccionario_datos)
print("diccionario_datos.md guardado ✅")

In [ ]:
# ============================================
# CELDA 6 — docs/diagrama_er.md
# ============================================
diagrama_er = '''# Diagrama ER e hipótesis de claves — Fase 3 / validación Fase 5

## Granularidad
- patients.csv: 1 fila = 1 paciente (10.000 filas)
- vitals_timeseries.csv / labs_timeseries.csv: 1 fila = 1 paciente en 1 hora

## Claves primarias
- patients: PK simple (patient_id)
- vitals / labs: PK compuesta (patient_id, hour_from_admission)

## Diagrama
patients (PK: patient_id)
   |-- 1:N --> vitals_timeseries (FK: patient_id)
   |-- 1:N --> labs_timeseries (FK: patient_id)
vitals_timeseries <-- (patient_id + hour coinciden) --> labs_timeseries : relación 1:1 confirmada

## Hipótesis planteadas en la Fase 3 y su validación real (Fase 5)
1. HIPOTESIS: labs tendría menos filas por paciente que vitals.
   REAL: ambas tablas dan exactamente 41.79 filas/paciente en promedio - la razón real
   es que ningún paciente llega a las 72h completas (los_hours varía).
2. HIPOTESIS: patient_id único y consistente entre las 3 tablas. REAL: confirmado (0/0).
3. HIPOTESIS: la columna de tiempo se llama "hour". REAL: se llama "hour_from_admission".
4. HIPOTESIS: relación 1:1 entre vitals y labs. REAL: confirmado.

## Hallazgo post-limpieza (Fase 7)
1231 filas excluidas de vitals por sistólica < diastólica. labs alineado a las mismas
claves. Panel final: 416.635 filas, 27 columnas.

## Archivos oficiales adicionales (confirmados en Fase 9)
hospital_deterioration_ml_ready.csv y hospital_deterioration_hourly_panel.csv incluyen
la columna deterioration_next_12h (target a nivel hora, ventana móvil) que no existe
en las tablas crudas originales.
'''

with open("docs/diagrama_er.md", "w") as f:
    f.write(diagrama_er)
print("diagrama_er.md guardado ✅")

In [ ]:
# ============================================
# CELDA 7 — docs/glosario_metricas.md
# ============================================
glosario_metricas = '''# Glosario de métricas de negocio — Fase 3b / corrección Fase 8

## Tasa de deterioro dentro de las 12h desde el ingreso (a nivel PACIENTE)
Formula: (pacientes con deterioration_within_12h_from_admission=1) / (total pacientes) x 100
Resultado real: 3.1%.

## Tasa de deterioro global (en toda la internación)
Formula: (pacientes con deterioration_event=1) / (total pacientes) x 100
Resultado real: 19.4%.

## Score de riesgo de sepsis (promedio y p90)
Formula: promedio(sepsis_risk_score) y percentil 90, agrupado por comorbilidad.

## Distribución de dispositivo de oxígeno
Formula: (filas con esa categoría) / (total de filas) x 100

## Advertencia de escala (hora vs. paciente)
No confundir métricas "a nivel hora" con "a nivel paciente" al reportar resultados.

## Correlación comorbidity_index vs baseline_risk_score (Fase 9)
r = 0.696 - correlación moderada-alta pero no redundante (no son la misma variable
medida dos veces); ambas aportan información parcialmente distinta.
'''

with open("docs/glosario_metricas.md", "w") as f:
    f.write(glosario_metricas)
print("glosario_metricas.md guardado ✅")

In [ ]:
# ============================================
# CELDA 8 — docs/auditoria_calidad.md
# ============================================
auditoria_calidad = '''# Auditoría de calidad de datos — Fase 6

## Nulos
0% de nulos en las tres tablas (patients, vitals, labs).

## Duplicados
0 duplicados de fila completa y de PK compuesta.

## Consistencia lógica
- Sistólica < diastólica: 1231 filas (error de carga, no caso real).
- oxygen_device="none" con oxygen_flow>0: 0 filas.
- deterioration_hour posterior a los_hours: 0 pacientes.

## Categóricas
gender: M/F. admission_type: ED 64.3%, Elective 25.9%, Transfer 9.8%.
oxygen_device: none/nasal/mask/hfnc/niv - todas limpias.

## Valores centinela
deterioration_hour=-1: valor intencional, "sin evento" (8062 pacientes, 80.6%).

## Clipping detectado (Fase 8)
Picos anómalos en los extremos de varias variables continuas - consistente con
límites de generación del dataset sintético.

## Multicolinealidad detectada (Fase 8)
lactate, creatinine, crp_level, wbc_count correlacionados entre sí (r>0.80) -
sugiere una variable oculta de "gravedad general" en el generador sintético.
'''

with open("docs/auditoria_calidad.md", "w") as f:
    f.write(auditoria_calidad)
print("auditoria_calidad.md guardado ✅")

In [ ]:
# ============================================
# CELDA 9 — docs/log_limpieza.md
# ============================================
log_limpieza = '''# Log de limpieza — Fase 7

## 1. Presión arterial inconsistente (vitals_timeseries)
Problema: 1231 filas (0.295%) con systolic_bp < diastolic_bp, imposible.
Decisión: exclusión. Justificación: error de carga, no caso clínico real.
Impacto: labs alineado con las mismas claves excluidas.

## 2. Valor centinela deterioration_hour = -1
Se mantiene sin modificar. NO se imputa ni se trata como nulo.

## 3. Nulos y duplicados
0% en ambos casos - no se requirió tratamiento.

## Resumen de volumen
| Tabla | Original | Limpio | % excluido |
|---|---|---|---|
| patients | 10.000 | 10.000 | 0% |
| vitals | 417.866 | 416.635 | 0.295% |
| labs | 417.866 | 416.635 | 0.295% (alineado) |

## Panel unido (Fase 8)
416.635 filas, 27 columnas. Guardado en data/processed/panel_clean.csv (solo Drive).
'''

with open("docs/log_limpieza.md", "w") as f:
    f.write(log_limpieza)
print("log_limpieza.md guardado ✅")

In [ ]:
# ============================================
# CELDA 10 — subir todo
# ============================================
!git add .
!git commit -m "docs: actualiza README y toda la documentación hasta Fase 8"
!git push origin main